# Análisis del Piloto UNSL — Template

Notebook de análisis estadístico para el capítulo empírico de la tesis.
Se alimenta del dataset exportado por `POST /analytics/cohort/export`.

**Entrada**: archivo JSON exportado (ej. `UNSL_2026_P2.json`).

**Estructura del análisis**:
1. Carga y exploración del dataset
2. Descriptivos a nivel episodio (distribución N4, coherencias)
3. Análisis longitudinal por estudiante
4. Correlaciones entre coherencias
5. Test de hipótesis (pre-post)
6. Análisis por cátedra (comparación P1, P2, TSU-IA)

**Nota ética**: el dataset ya viene anonymizado con salt. Guardar el `salt_hash`
en el repo (no el salt en claro) para reproducibilidad.

In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
sns.set_palette('colorblind')

# Seed para reproducibilidad
np.random.seed(2026)

## 1. Carga del dataset

Reemplazar `DATASET_PATH` con el archivo descargado del endpoint de export.

In [ ]:
DATASET_PATH = Path('UNSL_2026_P2.json')  # editar

with open(DATASET_PATH) as f:
    data = json.load(f)

print(f"Cohorte: {data['cohort_alias']}")
print(f"Exportado: {data['exported_at']}")
print(f"Schema version: {data['schema_version']}")
print(f"Salt hash (reproducibilidad): {data['salt_hash']}")
print()
print(f"Total episodios: {data['total_episodes']}")
print(f"Total estudiantes: {data['total_students']}")
print()
print('Distribución global:')
for label, count in data['distribution_summary'].items():
    print(f'  {label}: {count}')

In [ ]:
# Convertir a DataFrame para análisis
episodes = []
for ep in data['episodes']:
    row = {
        'episode_alias': ep['episode_alias'],
        'student_alias': ep['student_alias'],
        'opened_at': pd.to_datetime(ep['opened_at']) if ep['opened_at'] else None,
        'closed_at': pd.to_datetime(ep['closed_at']) if ep['closed_at'] else None,
        'duration_seconds': ep['duration_seconds'],
        'total_events': ep['total_events'],
        'appropriation': ep['appropriation'],
        'prompt_count': ep['event_counts']['prompts'],
        'code_exec_count': ep['event_counts']['code_executions'],
        'annotation_count': ep['event_counts']['annotations'],
        **ep['coherences'],
    }
    episodes.append(row)

df = pd.DataFrame(episodes)
print(f'Episodios cargados: {len(df)}')
df.head()

## 2. Descriptivos a nivel episodio

In [ ]:
# Distribución de categorías N4
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_counts = df['appropriation'].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values, 
            color=['#dc2626', '#f59e0b', '#16a34a', '#94a3b8'])
axes[0].set_title('Distribución de categorías N4')
axes[0].set_ylabel('# episodios')
axes[0].tick_params(axis='x', rotation=20)

# Torta relativa
axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
            colors=['#dc2626', '#f59e0b', '#16a34a', '#94a3b8'])
axes[1].set_title('Proporción de categorías N4')

plt.tight_layout()
plt.show()

In [ ]:
# Estadística descriptiva de coherencias por categoría
coherences = ['ct_summary', 'ccd_mean', 'ccd_orphan_ratio', 'cii_stability', 'cii_evolution']

print('Medias por categoría N4:')
print(df.groupby('appropriation')[coherences].mean().round(3))
print()
print('Desvíos estándar:')
print(df.groupby('appropriation')[coherences].std().round(3))

In [ ]:
# Boxplots de coherencias por categoría
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

category_order = ['delegacion_pasiva', 'apropiacion_superficial', 'apropiacion_reflexiva']

for idx, coh in enumerate(coherences):
    df_c = df[df['appropriation'].isin(category_order)].copy()
    df_c['appropriation'] = pd.Categorical(df_c['appropriation'], category_order, ordered=True)
    sns.boxplot(data=df_c, x='appropriation', y=coh, ax=axes[idx],
                palette=['#dc2626', '#f59e0b', '#16a34a'])
    axes[idx].set_title(f'{coh} por categoría N4')
    axes[idx].tick_params(axis='x', rotation=15)

axes[5].axis('off')
plt.tight_layout()
plt.show()

## 3. Análisis longitudinal por estudiante

Replica el análisis del endpoint `/cohort/{id}/progression` — mismos criterios
que el backend para consistencia.

In [ ]:
APPROPRIATION_ORDINAL = {
    'delegacion_pasiva': 0,
    'apropiacion_superficial': 1,
    'apropiacion_reflexiva': 2,
}

def progression_label(appropriations):
    """Replica platform_ops.longitudinal.StudentTrajectory.progression_label."""
    if len(appropriations) < 3:
        return 'insuficiente'
    scores = [APPROPRIATION_ORDINAL.get(a, -1) for a in appropriations if a in APPROPRIATION_ORDINAL]
    if len(scores) < 3:
        return 'insuficiente'
    n = len(scores)
    size = n // 3
    first = np.mean(scores[:size])
    last = np.mean(scores[-size:])
    if last - first > 0.25: return 'mejorando'
    if first - last > 0.25: return 'empeorando'
    return 'estable'

# Agrupar por estudiante y ordenar temporalmente
df_sorted = df.sort_values(['student_alias', 'opened_at'])
trajectories = (
    df_sorted.groupby('student_alias')
    .agg(
        n_episodes=('appropriation', 'count'),
        appropriation_sequence=('appropriation', list),
    )
)
trajectories['progression'] = trajectories['appropriation_sequence'].apply(progression_label)

print('Distribución de trayectorias:')
print(trajectories['progression'].value_counts())
print()

# Net progression ratio
counts = trajectories['progression'].value_counts()
suficientes = counts.get('mejorando', 0) + counts.get('estable', 0) + counts.get('empeorando', 0)
if suficientes > 0:
    ratio = (counts.get('mejorando', 0) - counts.get('empeorando', 0)) / suficientes
    print(f'Net progression ratio = {ratio:.4f}')
    if ratio > 0.3:
        print('→ Supera el umbral de éxito del protocolo (≥0.3).')
    else:
        print('→ No supera el umbral. Revisar el análisis cualitativo.')

In [ ]:
# Visualización de trayectorias individuales (heatmap)
max_episodes = df.groupby('student_alias').size().max()

# Construir matriz de aliases × episodios
matrix = []
aliases = []
for alias, group in df_sorted.groupby('student_alias'):
    seq = [APPROPRIATION_ORDINAL.get(a, np.nan) for a in group['appropriation']]
    seq = seq + [np.nan] * (max_episodes - len(seq))
    matrix.append(seq)
    aliases.append(alias)

# Ordenar por progresión final
order = sorted(range(len(matrix)), key=lambda i: np.nanmean(matrix[i][-3:]) if not np.all(np.isnan(matrix[i])) else -1)
matrix_ord = [matrix[i] for i in order]
aliases_ord = [aliases[i] for i in order]

fig, ax = plt.subplots(figsize=(14, max(6, len(aliases) * 0.15)))
im = ax.imshow(matrix_ord, aspect='auto', cmap='RdYlGn', vmin=0, vmax=2)
ax.set_xlabel('Nro. de episodio (cronológico)')
ax.set_ylabel('Estudiantes (ordenados por progresión)')
ax.set_title('Trayectorias N4 individuales')
cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(['delegación', 'superficial', 'reflexiva'])
plt.tight_layout()
plt.show()

## 4. Correlaciones entre coherencias

In [ ]:
corr = df[coherences].corr(method='pearson')

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, vmin=-1, vmax=1, ax=ax)
ax.set_title('Matriz de correlación de coherencias')
plt.tight_layout()
plt.show()

# Test de significancia de cada par
print('Correlaciones significativas (p < 0.05):')
for i, a in enumerate(coherences):
    for b in coherences[i+1:]:
        df_valid = df[[a, b]].dropna()
        if len(df_valid) < 3: continue
        r, p = stats.pearsonr(df_valid[a], df_valid[b])
        if p < 0.05:
            print(f'  {a} ↔ {b}: r={r:.3f}, p={p:.4f}')

## 5. Test de hipótesis pre-post

Comparamos el primer y último episodio de cada estudiante (con ≥3 episodios)
usando el test de McNemar (para categóricas pareadas).

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

# Filtrar estudiantes con >=3 episodios
eligible = trajectories[trajectories['n_episodes'] >= 3]
pre_post = []
for alias, row in eligible.iterrows():
    seq = row['appropriation_sequence']
    pre_post.append({'alias': alias, 'pre': seq[0], 'post': seq[-1]})

df_pp = pd.DataFrame(pre_post)
print(f'N estudiantes con pre/post: {len(df_pp)}')

# Tabla de contingencia 3x3
cats = ['delegacion_pasiva', 'apropiacion_superficial', 'apropiacion_reflexiva']
ct = pd.crosstab(df_pp['pre'], df_pp['post']).reindex(index=cats, columns=cats, fill_value=0)
print('\nTabla de contingencia pre × post:')
print(ct)

# McNemar con tabla binaria: reflexiva vs. el resto
df_pp['pre_reflexiva'] = (df_pp['pre'] == 'apropiacion_reflexiva')
df_pp['post_reflexiva'] = (df_pp['post'] == 'apropiacion_reflexiva')
ct_bin = pd.crosstab(df_pp['pre_reflexiva'], df_pp['post_reflexiva'])
print('\nTabla binaria reflexiva (pre × post):')
print(ct_bin)

if ct_bin.shape == (2, 2):
    result = mcnemar(ct_bin, exact=False, correction=True)
    print(f'\nMcNemar: statistic={result.statistic:.4f}, p={result.pvalue:.4f}')
    if result.pvalue < 0.05:
        print('→ Hay cambio significativo en proporción de reflexiva entre pre y post.')
    else:
        print('→ No se detecta cambio significativo.')

## 6. Resumen para reporte

Celda final que imprime métricas clave en formato copy-paste para la tesis.

In [ ]:
print('=' * 60)
print(f'RESUMEN — {data["cohort_alias"]}')
print('=' * 60)
print(f'Período: {data["period"]["from"]} → {data["period"]["to"]}')
print(f'N estudiantes: {data["total_students"]}')
print(f'N episodios: {data["total_episodes"]}')
print()
print('Distribución N4:')
total = data['total_episodes']
for cat in ['delegacion_pasiva', 'apropiacion_superficial', 'apropiacion_reflexiva', 'sin_clasificar']:
    count = data['distribution_summary'].get(cat, 0)
    pct = 100 * count / total if total > 0 else 0
    print(f'  {cat}: {count} ({pct:.1f}%)')
print()
print('Trayectorias longitudinales:')
tp_counts = trajectories['progression'].value_counts()
for cat, count in tp_counts.items():
    print(f'  {cat}: {count}')
print(f'Net progression ratio: {ratio:.4f}' if suficientes > 0 else 'Sin datos suficientes')
print()
print(f'Hash del salt de anonimización: {data["salt_hash"]}')
print('Guardar este hash para poder cruzar con análisis posteriores.')